In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/listings_features.csv")
print(df.shape)

features = [
    "accommodates", "bedrooms", "beds", "baths", "shared_bath",
    "latitude", "longitude", "dist_center_km",
    "review_scores_rating", "review_scores_location", "number_of_reviews",
    "has_reviews", "minimum_nights", "availability_365", "is_superhost",
    "n_amenities", "has_wifi", "has_ac", "has_kitchen", "has_balcony",
    "has_elevator", "has_parking",
]
cat_features = ["room_type", "neighbourhood_cleansed"]

X = df[features + cat_features].copy()
for c in cat_features:
    X[c] = X[c].astype("category")

y = np.log(df["price_num"])
groups = df["host_id"]

print(X.shape, y.shape, groups.nunique())

(13832, 103)
(13832, 24) (13832,) 5316


In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(len(X_train), len(X_test))

# Baseline: Ridge με one-hot για τις κατηγορικές και συμπλήρωση κενών
X_lin = pd.get_dummies(X, columns=cat_features, dtype=float)
X_lin = X_lin.fillna(X_lin.median())
Xl_train, Xl_test = X_lin.iloc[train_idx], X_lin.iloc[test_idx]

ridge = Ridge(alpha=1.0).fit(Xl_train, y_train)
pred = ridge.predict(Xl_test)

print("R² (log):", round(r2_score(y_test, pred), 3))
print("MAE (σε τιμή):", round(mean_absolute_error(np.exp(y_test), np.exp(pred)), 1))

11177 2655
R² (log): 0.484
MAE (σε τιμή): 35.7


In [3]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)

pred_lgb = model.predict(X_test)
print("R² (log):", round(r2_score(y_test, pred_lgb), 3))
print("MAE (σε τιμή):", round(mean_absolute_error(np.exp(y_test), np.exp(pred_lgb)), 1))

c:\Users\paulp\Desktop\Project\VisitAth\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's l2: 0.101717
Early stopping, best iteration is:
[269]	valid_0's l2: 0.100853
R² (log): 0.576
MAE (σε τιμή): 33.1


In [4]:
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(10))

dist_center_km            769
availability_365          764
n_amenities               745
neighbourhood_cleansed    683
number_of_reviews         682
latitude                  611
longitude                 568
review_scores_rating      547
minimum_nights            532
review_scores_location    452
dtype: int32


In [5]:
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(10))

dist_center_km            769
availability_365          764
n_amenities               745
neighbourhood_cleansed    683
number_of_reviews         682
latitude                  611
longitude                 568
review_scores_rating      547
minimum_nights            532
review_scores_location    452
dtype: int32


In [6]:
test = df.iloc[test_idx][["name", "neighbourhood_cleansed", "accommodates", "price_num", "review_scores_rating"]].copy()
test["fair_price"] = np.exp(pred_lgb)
test["value_score"] = (test["fair_price"] - test["price_num"]) / test["fair_price"]

cols = ["name", "neighbourhood_cleansed", "accommodates", "price_num", "fair_price", "value_score", "review_scores_rating"]
print(test.sort_values("value_score", ascending=False).head(10)[cols])
print(test.sort_values("value_score").head(5)[cols])

                                                   name  \
239                 LargeBeautifulVeryCleanCentral Flat   
797                                    Νεοκλασική άποψη   
13273             1# Central Economy Student Studio ¶₹ቢ   
7480     Filoxenia MG Projects Acropolis Dimitrakopulou   
13277               3# Central Economy Student Studio †   
11657  Central 2-BR Apt near Acropolis/Plaka/Metro/Tram   
9759                        Under the Rock of Acropolis   
12587       Vintage En-Suite Δωμάτιο στους Αμπελοκήπους   
12387                       Athens Urban Haven Exarchia   
10912             Minimal Urban apartment near Kolonaki   

         neighbourhood_cleansed  accommodates  price_num  fair_price  \
239                    ΚΟΛΩΝΑΚΙ             5      37.36  103.976206   
797                  ΛΥΚΑΒΗΤΤΟΣ             4      57.20  154.905812   
13273    ΕΜΠΟΡΙΚΟ ΤΡΙΓΩΝΟ-ΠΛΑΚΑ             4      36.00   95.546047   
7480        ΚΟΥΚΑΚΙ-ΜΑΚΡΥΓΙΑΝΝΗ             4      58.50  154.

In [7]:
good = test[
    (test["review_scores_rating"] >= 4.5)
    & (test["value_score"] > 0.10)
    & (test["value_score"] < 0.50)
].copy()

# προσθέτουμε τον αριθμό κριτικών από το df
good["number_of_reviews"] = df.loc[good.index, "number_of_reviews"]
good = good[good["number_of_reviews"] >= 10]

print(len(good))
print(good.sort_values("value_score", ascending=False).head(10)[
    ["name", "neighbourhood_cleansed", "accommodates", "price_num", "fair_price", "value_score", "review_scores_rating", "number_of_reviews"]
])

649
                                                    name  \
4674                      Central flat next to Acropolis   
13310               Modern flat next to Acropolis Museum   
4586         Filoxenia MG Projects Acropolis Diovounioti   
335                              历史与当代结合-一百平米独栋大屋-超近雅典卫城   
13138  Christina s Cozy Apartment Plaka Acropolis Athens   
1174                         Athens Heart Cosy Apartment   
13060                          OTH Grand View King Suite   
2205                                    Ασφαλής Γειτονιά   
3141   A28 2' walk 2 Acropolis Museum!2BR. Sunny terrace   
12812                                   Friends & Family   

                 neighbourhood_cleansed  accommodates  price_num  fair_price  \
4674                ΚΟΥΚΑΚΙ-ΜΑΚΡΥΓΙΑΝΝΗ             6     143.00  285.877947   
13310               ΚΟΥΚΑΚΙ-ΜΑΚΡΥΓΙΑΝΝΗ             4      91.20  179.843290   
4586                ΚΟΥΚΑΚΙ-ΜΑΚΡΥΓΙΑΝΝΗ             4      72.60  142.796146   
335            

In [8]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(df))
best_iters = []

for tr, va in gkf.split(X, y, groups):
    m = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        min_child_samples=20, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, random_state=42, verbose=-1,
    )
    m.fit(X.iloc[tr], y.iloc[tr],
          eval_set=[(X.iloc[va], y.iloc[va])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof[va] = m.predict(X.iloc[va])
    best_iters.append(m.best_iteration_)

print("OOF R² (log):", round(r2_score(y, oof), 3))
print("OOF MAE ($):", round(mean_absolute_error(np.exp(y), np.exp(oof)), 1))
print("Best iterations:", best_iters)

df["fair_price"] = np.exp(oof)
df["value_score"] = (df["fair_price"] - df["price_num"]) / df["fair_price"]
df[["id", "name", "price_num", "fair_price", "value_score"]].describe()

c:\Users\paulp\Desktop\Project\VisitAth\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\paulp\Desktop\Project\VisitAth\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\paulp\Desktop\Project\VisitAth\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\paulp\Desktop\Project\VisitAth\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' inste

OOF R² (log): 0.576
OOF MAE ($): 35.7
Best iterations: [246, 360, 185, 268, 621]


,id,price_num,fair_price,value_score
count,1.383200e+04,13832.000000,13832.000000,13832.000000
mean,8.992171e+17,135.212534,127.100072,-0.061347
std,6.170279e+17,84.885851,57.384902,0.439216
min,3.394500e+04,36.000000,33.752231,-7.166847
25%,5.184989e+07,82.000000,90.408821,-0.197111
50%,1.084372e+18,111.625000,113.413431,0.025408
75%,1.407033e+18,158.000000,146.223249,0.192651
max,1.779798e+18,672.030000,608.341495,0.744437


In [9]:
df.to_csv("../data/processed/listings_scored.csv", index=False)